In [1]:
#Task 1: Data Preprocessing and Model Training
import os
import sys
import platform
import joblib
import numpy as np
import pandas as pd
import sklearn

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

OUTPUT_DIR = "../outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

adult = fetch_openml("adult", version=2, as_frame=True, parser="auto")
df = adult.frame.copy()
df = df.replace("?", np.nan)

df["target"] = df["class"].map({"<=50K": 0, ">50K": 1}).astype(int)
df = df.drop(columns=["class"])

def create_engineered_features(data):
    data = data.copy()
    data["age_bucket"] = pd.cut(data["age"], bins=[0, 25, 35, 45, 55, 65, np.inf], labels=["Young", "Early_Career", "Mid_Career", "Experienced", "Senior", "Older"])
    data["hours_bucket"] = pd.cut(data["hours-per-week"], bins=[0, 30, 40, 50, 60, np.inf], labels=["Part_Time", "Standard", "Overtime", "High_Hours", "Very_High_Hours"])
    data["capital_gain_flag"] = (data["capital-gain"] > 0).astype(int)
    data["log_capital_gain"] = np.log1p(data["capital-gain"])
    data["higher_education"] = (data["education-num"] >= 13).astype(int)
    data["education_hours_interaction"] = data["education-num"] * data["hours-per-week"]
    data["capital_loss_flag"] = (data["capital-loss"] > 0).astype(int)
    data["age_hours_interaction"] = data["age"] * data["hours-per-week"]
    return data

X = df.drop(columns=["target"])
y = df["target"]

numeric_features = ["age", "fnlwgt", "education-num", "capital-gain", "capital-loss", "hours-per-week", "capital_gain_flag", "log_capital_gain", "higher_education", "education_hours_interaction", "capital_loss_flag", "age_hours_interaction"]
categorical_features = ["workclass", "education", "marital-status", "occupation", "relationship", "race", "sex", "native-country", "age_bucket", "hours_bucket"]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

logistic_pipeline = Pipeline([
    ("feature_engineering", FunctionTransformer(create_engineered_features, validate=False)),
    ("preprocessing", preprocessor),
    ("model", LogisticRegression(solver="liblinear", max_iter=1000, random_state=RANDOM_STATE))
])
random_forest_pipeline = Pipeline([
    ("feature_engineering", FunctionTransformer(create_engineered_features, validate=False)),
    ("preprocessing", preprocessor),
    ("model", RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=1))
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
logistic_pipeline.fit(X_train, y_train)
random_forest_pipeline.fit(X_train, y_train)

logistic_predictions = logistic_pipeline.predict(X_test)
random_forest_predictions = random_forest_pipeline.predict(X_test)
results = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest"],
    "Test Accuracy": [accuracy_score(y_test, logistic_predictions), accuracy_score(y_test, random_forest_predictions)]
})
display(results)

joblib.dump(logistic_pipeline, os.path.join(OUTPUT_DIR, "logistic_pipeline.joblib"))
joblib.dump(random_forest_pipeline, os.path.join(OUTPUT_DIR, "random_forest_pipeline.joblib"))

version_info = pd.DataFrame({
    "Component": ["Python", "Platform", "NumPy", "Pandas", "Scikit-learn", "Joblib"],
    "Version": [sys.version, platform.platform(), np.__version__, pd.__version__, sklearn.__version__, joblib.__version__]
})
version_info.to_csv(os.path.join(OUTPUT_DIR, "library_versions.csv"), index=False)

loaded_logistic = joblib.load(os.path.join(OUTPUT_DIR, "logistic_pipeline.joblib"))
loaded_random_forest = joblib.load(os.path.join(OUTPUT_DIR, "random_forest_pipeline.joblib"))
print("Logistic reload verification:", np.array_equal(logistic_pipeline.predict(X_test), loaded_logistic.predict(X_test)))
print("Random Forest reload verification:", np.array_equal(random_forest_pipeline.predict(X_test), loaded_random_forest.predict(X_test)))
print("Saved files:")
print("logistic_pipeline.joblib")
print("random_forest_pipeline.joblib")
print("library_versions.csv")

,Model,Test Accuracy
0,Logistic Regression,0.859760
1,Random Forest,0.851981


Logistic reload verification: True
Random Forest reload verification: True
Saved files:
logistic_pipeline.joblib
random_forest_pipeline.joblib
library_versions.csv


In [ ]:
# Task2 
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import f1_score, accuracy_score
from scipy.stats import loguniform, randint

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

logistic_params = {
    "model__C": loguniform(0.001, 100),
    "model__l1_ratio": [0, 1]
}

random_forest_params = {
    "model__n_estimators": randint(100, 500),
    "model__max_depth": [None, 5, 10, 15, 20, 30, 40],
    "model__min_samples_leaf": randint(1, 10),
    "model__max_features": ["sqrt", "log2", 0.5, 0.75]
}

logistic_pipeline.set_params(
    model__solver="saga",
    model__penalty="elasticnet"
)

logistic_search = RandomizedSearchCV(
    estimator=logistic_pipeline,
    param_distributions=logistic_params,
    n_iter=20,
    scoring="f1",
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1,
    refit=True
)

random_forest_search = RandomizedSearchCV(
    estimator=random_forest_pipeline,
    param_distributions=random_forest_params,
    n_iter=20,
    scoring="f1",
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1,
    refit=True
)

print("Tuning Logistic Regression...")
logistic_search.fit(X_train, y_train)

print("\nTuning Random Forest...")
random_forest_search.fit(X_train, y_train)

best_logistic_model = logistic_search.best_estimator_
best_random_forest_model = random_forest_search.best_estimator_

logistic_test_predictions = best_logistic_model.predict(X_test)
random_forest_test_predictions = best_random_forest_model.predict(X_test)

logistic_test_f1 = f1_score(y_test, logistic_test_predictions)
random_forest_test_f1 = f1_score(y_test, random_forest_test_predictions)

logistic_test_accuracy = accuracy_score(y_test, logistic_test_predictions)
random_forest_test_accuracy = accuracy_score(y_test, random_forest_test_predictions)

print("\nLOGISTIC REGRESSION")
print("Best CV F1:", logistic_search.best_score_)
print("Best Parameters:", logistic_search.best_params_)
print("Test F1:", logistic_test_f1)
print("Test Accuracy:", logistic_test_accuracy)

print("\nRANDOM FOREST")
print("Best CV F1:", random_forest_search.best_score_)
print("Best Parameters:", random_forest_search.best_params_)
print("Test F1:", random_forest_test_f1)
print("Test Accuracy:", random_forest_test_accuracy)

task2_results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest"
    ],
    "Best CV F1": [
        logistic_search.best_score_,
        random_forest_search.best_score_
    ],
    "Test F1": [
        logistic_test_f1,
        random_forest_test_f1
    ],
    "Test Accuracy": [
        logistic_test_accuracy,
        random_forest_test_accuracy
    ]
})

display(task2_results)

joblib.dump(
    best_logistic_model,
    os.path.join(OUTPUT_DIR, "best_logistic_pipeline.joblib")
)

joblib.dump(
    best_random_forest_model,
    os.path.join(OUTPUT_DIR, "best_random_forest_pipeline.joblib")
)


Tuning Logistic Regression...
Fitting 5 folds for each of 20 candidates, totalling 100 fits


c:\Users\samir\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(



Tuning Random Forest...
Fitting 5 folds for each of 20 candidates, totalling 100 fits

LOGISTIC REGRESSION
Best CV F1: 0.6745280279960943
Best Parameters: {'model__C': np.float64(73.92266140516048), 'model__l1_ratio': 0}
Test F1: 0.6744020838266636
Test Accuracy: 0.8592486436687481

RANDOM FOREST
Best CV F1: 0.6838025483331801
Best Parameters: {'model__max_depth': 15, 'model__max_features': 0.5, 'model__min_samples_leaf': 7, 'model__n_estimators': 363}
Test F1: 0.6906302420321112
Test Accuracy: 0.8678472719828028


,Model,Best CV F1,Test F1,Test Accuracy
0,Logistic Regression,0.674528,0.674402,0.859249
1,Random Forest,0.683803,0.690630,0.867847



Task 2 completed successfully.
Saved:
best_logistic_pipeline.joblib
best_random_forest_pipeline.joblib


In [ ]:
# Task 3: Learning Curves and Regularization Strength Analysis
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import learning_curve
from sklearn.metrics import f1_score

models = {
    "Logistic Regression": best_logistic_model,
    "Random Forest": best_random_forest_model
}

train_sizes = np.linspace(0.1, 1.0, 8)

learning_curve_results = {}

for name, model in models.items():
    sizes, train_scores, validation_scores = learning_curve(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring="f1",
        train_sizes=train_sizes,
        n_jobs=-1
    )

    train_mean = train_scores.mean(axis=1)
    train_std = train_scores.std(axis=1)
    validation_mean = validation_scores.mean(axis=1)
    validation_std = validation_scores.std(axis=1)

    learning_curve_results[name] = {
        "sizes": sizes,
        "train_mean": train_mean,
        "train_std": train_std,
        "validation_mean": validation_mean,
        "validation_std": validation_std
    }

    plt.figure(figsize=(9, 6))
    plt.plot(sizes, train_mean, marker="o", label="Training F1")
    plt.plot(sizes, validation_mean, marker="o", label="Validation F1")
    plt.fill_between(
        sizes,
        train_mean - train_std,
        train_mean + train_std,
        alpha=0.15
    )
    plt.fill_between(
        sizes,
        validation_mean - validation_std,
        validation_mean + validation_std,
        alpha=0.15
    )
    plt.xlabel("Training Set Size")
    plt.ylabel("F1 Score")
    plt.title(f"Learning Curve - {name}")
    plt.legend()
    plt.grid(True)
    plt.show()

C_values = [0.001, 0.01, 0.1, 1, 10, 100]

C_results = []

for C in C_values:
    model = joblib.clone(best_logistic_model) if hasattr(joblib, "clone") else None

    from sklearn.base import clone

    model = clone(best_logistic_model)
    model.set_params(model__C=C)

    model.fit(X_train, y_train)

    train_predictions = model.predict(X_train)
    validation_predictions = model.predict(X_test)

    C_results.append({
        "C": C,
        "Training F1": f1_score(y_train, train_predictions),
        "Test F1": f1_score(y_test, validation_predictions)
    })

C_results = pd.DataFrame(C_results)

display(C_results)

plt.figure(figsize=(9, 6))
plt.semilogx(
    C_results["C"],
    C_results["Training F1"],
    marker="o",
    label="Training F1"
)
plt.semilogx(
    C_results["C"],
    C_results["Test F1"],
    marker="o",
    label="Test F1"
)
plt.xlabel("C")
plt.ylabel("F1 Score")
plt.title("Effect of Regularization Strength on Logistic Regression")
plt.legend()
plt.grid(True)
plt.show()

best_c_row = C_results.loc[C_results["Test F1"].idxmax()]

print("Best C:", best_c_row["C"])
print("Best Test F1:", best_c_row["Test F1"])


In [ ]:
# Task 4: Probability Calibration and Threshold Selection
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import (
    brier_score_loss,
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
    confusion_matrix
)
from sklearn.model_selection import StratifiedKFold, cross_val_predict

final_model = best_logistic_model

calibrated_model = CalibratedClassifierCV(
    estimator=final_model,
    method="sigmoid",
    cv=5,
    n_jobs=-1
)

calibrated_model.fit(X_train, y_train)

test_probabilities = calibrated_model.predict_proba(X_test)[:, 1]

brier_score = brier_score_loss(y_test, test_probabilities)

prob_true, prob_pred = calibration_curve(
    y_test,
    test_probabilities,
    n_bins=10,
    strategy="uniform"
)

plt.figure(figsize=(8, 6))
plt.plot(
    prob_pred,
    prob_true,
    marker="o",
    label="Calibrated Model"
)
plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Perfect Calibration"
)
plt.xlabel("Mean Predicted Probability")
plt.ylabel("Fraction of Positives")
plt.title("Probability Calibration Curve")
plt.legend()
plt.grid(True)
plt.show()

print("Brier Score:", brier_score)

oof_probabilities = cross_val_predict(
    calibrated_model,
    X_train,
    y_train,
    cv=cv,
    method="predict_proba",
    n_jobs=-1
)[:, 1]

thresholds = np.arange(0.10, 0.91, 0.01)

threshold_results = []

for threshold in thresholds:
    predictions = (oof_probabilities >= threshold).astype(int)

    threshold_results.append({
        "Threshold": threshold,
        "F1": f1_score(y_train, predictions),
        "Precision": precision_score(y_train, predictions, zero_division=0),
        "Recall": recall_score(y_train, predictions, zero_division=0)
    })

threshold_results = pd.DataFrame(threshold_results)

best_threshold_row = threshold_results.loc[
    threshold_results["F1"].idxmax()
]

best_threshold = best_threshold_row["Threshold"]

display(threshold_results.sort_values("F1", ascending=False).head(10))

plt.figure(figsize=(9, 6))
plt.plot(
    threshold_results["Threshold"],
    threshold_results["F1"],
    label="F1"
)
plt.plot(
    threshold_results["Threshold"],
    threshold_results["Precision"],
    label="Precision"
)
plt.plot(
    threshold_results["Threshold"],
    threshold_results["Recall"],
    label="Recall"
)
plt.axvline(
    best_threshold,
    linestyle="--",
    label=f"Best Threshold = {best_threshold:.2f}"
)
plt.xlabel("Classification Threshold")
plt.ylabel("Score")
plt.title("Threshold Selection")
plt.legend()
plt.grid(True)
plt.show()

default_predictions = (test_probabilities >= 0.50).astype(int)
tuned_predictions = (test_probabilities >= best_threshold).astype(int)

default_cm = confusion_matrix(y_test, default_predictions)
tuned_cm = confusion_matrix(y_test, tuned_predictions)

threshold_comparison = pd.DataFrame({
    "Metric": [
        "F1",
        "Precision",
        "Recall",
        "Accuracy"
    ],
    "Threshold 0.50": [
        f1_score(y_test, default_predictions),
        precision_score(y_test, default_predictions, zero_division=0),
        recall_score(y_test, default_predictions, zero_division=0),
        accuracy_score(y_test, default_predictions)
    ],
    f"Threshold {best_threshold:.2f}": [
        f1_score(y_test, tuned_predictions),
        precision_score(y_test, tuned_predictions, zero_division=0),
        recall_score(y_test, tuned_predictions, zero_division=0),
        accuracy_score(y_test, tuned_predictions)
    ]
})

display(threshold_comparison)

print("Default Threshold Confusion Matrix:")
print(default_cm)

print("\nTuned Threshold Confusion Matrix:")
print(tuned_cm)

print("\nBest Threshold:", round(best_threshold, 2))
print("Best Training OOF F1:", round(best_threshold_row["F1"], 4))
print("Test Brier Score:", round(brier_score, 4))

final_artifact = {
    "model": calibrated_model,
    "threshold": float(best_threshold),
    "brier_score": float(brier_score)
}

joblib.dump(
    final_artifact,
    os.path.join(OUTPUT_DIR, "final_calibrated_model.joblib")
)


In [ ]:
# Task 5: Final Model Evaluation and Saving
import os
import joblib
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    brier_score_loss,
    confusion_matrix,
    classification_report
)

final_predictions = (
    test_probabilities >= best_threshold
).astype(int)

final_accuracy = accuracy_score(y_test, final_predictions)
final_precision = precision_score(
    y_test,
    final_predictions,
    zero_division=0
)
final_recall = recall_score(
    y_test,
    final_predictions,
    zero_division=0
)
final_f1 = f1_score(
    y_test,
    final_predictions,
    zero_division=0
)
final_roc_auc = roc_auc_score(
    y_test,
    test_probabilities
)
final_brier = brier_score_loss(
    y_test,
    test_probabilities
)

final_metrics = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score",
        "ROC-AUC",
        "Brier Score"
    ],
    "Score": [
        final_accuracy,
        final_precision,
        final_recall,
        final_f1,
        final_roc_auc,
        final_brier
    ]
})

display(final_metrics)

print("Confusion Matrix:")
print(confusion_matrix(y_test, final_predictions))

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        final_predictions,
        target_names=["<=50K", ">50K"],
        zero_division=0
    )
)

final_artifact = {
    "model": calibrated_model,
    "threshold": float(best_threshold),
    "metrics": {
        "accuracy": float(final_accuracy),
        "precision": float(final_precision),
        "recall": float(final_recall),
        "f1": float(final_f1),
        "roc_auc": float(final_roc_auc),
        "brier_score": float(final_brier)
    },
    "best_threshold": float(best_threshold),
    "random_state": RANDOM_STATE
}

final_model_path = os.path.join(
    OUTPUT_DIR,
    "final_adult_income_pipeline.joblib"
)

joblib.dump(
    final_artifact,
    final_model_path
)

def predict_income(data):
    probabilities = final_artifact["model"].predict_proba(data)[:, 1]
    predictions = (
        probabilities >= final_artifact["threshold"]
    ).astype(int)

    return pd.DataFrame({
        "Predicted_Income": np.where(
            predictions == 1,
            ">50K",
            "<=50K"
        ),
        "Probability_Above_50K": probabilities
    })

sample_predictions = predict_income(X_test.iloc[:10])

display(sample_predictions)

print("Final model saved successfully.")
print("Path:", final_model_path)
print("Selected threshold:", round(best_threshold, 4))
print("Final F1 Score:", round(final_f1, 4))
print("Final Accuracy:", round(final_accuracy, 4))
print("Final ROC-AUC:", round(final_roc_auc, 4))
print("Final Brier Score:", round(final_brier, 4))
